# 06a: Cross-Ablation — Configs A & B (SiLU variants)

**Part 1 of 2** for the activation × attention cross-ablation.
Runs Config A (SiLU+SE) and Config B (SiLU+ECA), 3 seeds each = 6 runs.

**Estimated time:** ~31h on T4. If on Kaggle (12h limit), run 2 seeds per session.

In [ ]:
import os, sys, socket
try:
    socket.create_connection(("github.com", 80), timeout=5)
except:
    print("\u274c ERROR: No Internet connection.")
    sys.exit(1)

!rm -rf tinyYOLO && git clone https://github.com/ShMazumder/tinyYOLO.git
%cd tinyYOLO
!pip install -e . -q
!pip install tqdm timm -q
!python3 -c "from ultralytics.data.utils import check_det_dataset; check_det_dataset('VOC.yaml')"

In [ ]:
from tinyYOLO.models import build_model
import torch
x = torch.randn(1, 3, 416, 416)
for v, a, l in [('standard','default','A: SiLU+SE'), ('standard','eca','B: SiLU+ECA')]:
    m, i = build_model(task='det', variant=v, nc=20, attention=a)
    _ = m(x)
    print(f'  \u2713 {l}: {i["total_params_M"]}M, attn3={m.backbone.attn3.__class__.__name__}')
print('Verified!')

In [ ]:
# Config A: SiLU + SE/Spatial (= TinyYOLO-std baseline, should reproduce ~38.7%)
SEEDS = [42, 123, 256]
for s in SEEDS:
    !python3 scripts/train.py --task det --variant standard --attention default \
        --data voc.yaml --imgsz 416 --epochs 300 --batch 128 --seed {s} \
        --warmup 3 --pretrained --compile --name xabl-A-silu-se-seed{s}

In [ ]:
# Config B: SiLU + ECA (cross — isolates ECA effect with SiLU)
SEEDS = [42, 123, 256]
for s in SEEDS:
    !python3 scripts/train.py --task det --variant standard --attention eca \
        --data voc.yaml --imgsz 416 --epochs 300 --batch 128 --seed {s} \
        --warmup 3 --pretrained --compile --name xabl-B-silu-eca-seed{s}